# MetabTravLR — submit SpaceTravLR runs to SLURM

One cell per dataset. Running a cell submits a **single SLURM job** that does the whole
thing for that dataset — setup, training, and artifacts — and returns immediately with a
job id. Nothing heavy happens in this notebook.

What the job does (`run_spacetravlr.py`):

| stage | writes |
|---|---|
| `setup` | `spacetravlr_output/input_data/` (processed adata, CellOracle links, NicheNet links) |
| `fit` | `spacetravlr_output/betadata/<gene>_betadata.parquet` |
| `artifacts` | `easy_download/metabtravlr_outputs/<tier>/{gene_pairs.csv, histograms.csv, histograms.png}` and `spacetravlr_adata.h5ad` |

**Logs** go to `{METAB_DATA_DIR}/spacetravlr_logs/<DATASET>/<stages>_<timestamp>.log` — a
sibling of `harreman_logs/`, deliberately *outside* `spacetravlr_output/`. SLURM opens
that file before the job body runs, so it cannot live in the directory the job is about
to create.

**Reruns are cheap and safe.** Setup is skipped when it's already complete, and `fit` skips
any gene that already has a betadata parquet — so re-submitting after a timeout resumes
rather than starting over.

Per-dataset settings (cell-type column, tiers, target genes, SLURM resources) live in
`dataset_configs.py`. Target genes are shared across datasets by default
(`metab_travlr_config.FOCUS_GENES`); metabolite pairs always come from each dataset's own
harreman `metabolite_selection.yaml`.

In [ ]:
import sys
from pathlib import Path
# Make the repo root importable regardless of CWD or machine: walk up from the
# working dir until we hit a repo marker, then put that dir on sys.path.
_start = Path.cwd()
_root = next((p for p in (_start, *_start.parents)
              if (p / ".git").exists() or (p / "setup.py").exists()), _start)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from metab_processing.SpaceTravLR.submit_spacetravlr import submit
from metab_processing.SpaceTravLR.dataset_configs import DATASETS, get_config

print('datasets:', sorted(DATASETS))
print('target genes:', get_config(sorted(DATASETS)[0])['focus_genes'])

## Primary_Dermal_Melanoma

In [ ]:
submit('Primary_Dermal_Melanoma')

## Human_Lung

In [ ]:
submit('Human_Lung')

## Variations

`submit()` takes:

- `stages=['setup'|'fit'|'artifacts']` — run only part of the pipeline.
- `overwrite=True` — delete `input_data/` and redo setup. Needs the `setup` stage (it is
  rejected rather than silently ignored otherwise). **Trained betadata is kept**, so those
  betas came from the *previous* preprocessing — the job log says so.
- `clear_betadata=True` — delete `betadata/`, forcing every gene to retrain. Independent of
  `overwrite`; combine the two for a genuinely clean slate.
- `dry_run=True` — print the sbatch settings and command without submitting.
- any SLURM key as a keyword (`time_hours`, `partition`, `qos`, `gres`, `cpus_per_task`,
  `account`, `python_path`) to override the dataset's config for this submission only.

Two jobs may `fit` the same dataset concurrently (the gene queue is lock-based), but **not
`setup`** — a second setup on a dataset already being set up refuses, via a `.setup.lock`
in `spacetravlr_output/`. If a setup job was killed, delete that file before resubmitting.

In [ ]:
# See exactly what would be submitted, without submitting.
submit('Human_Lung', dry_run=True)

In [ ]:
# Re-do just the read-out after a finished training run (CPU-only, minutes not hours).
# submit('Primary_Dermal_Melanoma', stages=['artifacts'], time_hours=2)

# Redo setup from scratch and retrain every gene.
# submit('Human_Lung', overwrite=True, clear_betadata=True, time_hours=24)

# Spawn a second worker on a dataset already training -- the gene queue is lock-based,
# so extra workers just pick up untrained genes.
# submit('Human_Lung', stages=['fit'])

## Monitor

In [ ]:
!squeue -u $USER

In [ ]:
# Tail the newest log for a dataset.
from metab_processing.SpaceTravLR.dataset_configs import dataset_paths

DATASET = 'Primary_Dermal_Melanoma'
logs = sorted(dataset_paths(DATASET)['log_dir'].glob('*.log'))
print(logs[-1] if logs else 'no logs yet')
if logs:
    print(''.join(logs[-1].read_text().splitlines(keepends=True)[-40:]))

In [ ]:
# What has finished training so far.
for dataset in sorted(DATASETS):
    paths = dataset_paths(dataset)
    done = sorted(p.name[:-len('_betadata.parquet')]
                  for p in paths['betadata'].glob('*_betadata.parquet')) \
        if paths['betadata'].exists() else []
    print(f"{dataset:28s} setup={paths['input_data'].is_dir()}  "
          f"beta_adata={paths['beta_adata'].exists()}  genes={done}")

## Publish

Copy each dataset's `easy_download/` (now including `metabtravlr_outputs/`) into the
aggregate `Results/` tree — the same step `quick_start_metab.ipynb` and `run_full_harr.ipynb`
end with. Deliberately **not** part of the job: it touches every dataset, not just the one
that ran, so it's a manual step once the runs you care about have finished.

In [ ]:
sys.path.insert(0, str(_root / 'metab_processing'))
from Harreman.copy_easy_download import save_easy_downloads

save_easy_downloads()